# KD-Trees (Árvores k-Dimensionais) na Célula de Manufatura Flexível
### Indexação Espacial 2D (`AGV-300` / RRT `NEAREST`) • Busca por Alcance • Classificação RGB 3D (`INSP-200`)
**Disciplina:** ECAA08 • Engenharia de Controle e Automação & Matemática Discreta  
**Unidade de Processo:** Célula de Manufatura Flexível (FMS) — Layout $\mathcal{C} = [0, 5{,}0\text{ m}] \times [0, 4{,}0\text{ m}]$ e Túnel Óptico `INSP-200`  

---

## 🎯 Objetivos do Notebook
1. **Implementação Algorítmica da KD-Tree:** Construir do zero uma `KDTree` genérica para $\mathbb{R}^k$ com particionamento por mediana, busca de vizinho mais próximo (*Nearest Neighbor* com poda *branch-and-bound*), $k$-NN e busca por alcance (*Range Search*).
2. **Aplicação 1 (2D Espacial — `AGV-300` & Aula 22):** Indexar as estações e *waypoints* livres da célula FMS ($\mathcal{C} = [0, 5{,}0] \times [0, 4{,}0]\text{ m}$), visualizar os hiperplanos de corte e o círculo de poda.
3. **Aplicação 2 (3D RGB — `INSP-200`):** Classificar leituras contínuas $(R, G, B)$ do túnel óptico em `CX-301` (Vermelha/`XV-201`), `CX-302` (Verde/`XV-202`), `CX-303` (Azul/`XV-203`) ou `DESC-304` (Refugo).
4. **Auditoria de Exatidão (`assert`):** Comprovar que a KD-Tree retorna exatamente o mesmo vizinho mais próximo que a busca linear por força bruta.
5. **Benchmark de Desempenho:** Comparar nós visitados e tempo de execução entre Força Bruta $O(N)$ e KD-Tree $O(\log N)$ para $N \in [100, 10000]$.

In [ ]:
import time
import heapq
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from typing import List, Tuple, Optional, Any

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

print("✅ Módulos carregados para análise e simulação de KD-Trees na FMS.")

---
## 1. Implementação da KD-Tree Genérica em $\mathbb{R}^k$

In [ ]:
class KDNode:
    """Nó da KD-Tree armazenando ponto k-dimensional, rótulo/payload e subárvores."""
    def __init__(self, point: np.ndarray, label: Any, axis: int, left=None, right=None):
        self.point = point
        self.label = label
        self.axis = axis
        self.left: Optional[KDNode] = left
        self.right: Optional[KDNode] = right


class KDTree:
    """Árvore k-Dimensional com particionamento por mediana e poda branch-and-bound."""
    def __init__(self, points: List[Tuple[np.ndarray, Any]], k: int):
        self.k = k
        self.root = self._build(points, depth=0)

    def _build(self, points: List[Tuple[np.ndarray, Any]], depth: int) -> Optional[KDNode]:
        if not points:
            return None
        axis = depth % self.k
        points_sorted = sorted(points, key=lambda item: item[0][axis])
        mid = len(points_sorted) // 2
        pt, lbl = points_sorted[mid]
        return KDNode(
            point=np.asarray(pt, dtype=float),
            label=lbl,
            axis=axis,
            left=self._build(points_sorted[:mid], depth + 1),
            right=self._build(points_sorted[mid + 1:], depth + 1),
        )

    def nearest_neighbor(self, query: np.ndarray) -> Tuple[Optional[KDNode], float, int]:
        """Retorna (melhor_no, distancia_euclidiana, nos_visitados)."""
        query = np.asarray(query, dtype=float)
        best = {"node": None, "dist2": float("inf"), "visited": 0}

        def _search(node: Optional[KDNode]):
            if node is None:
                return
            best["visited"] += 1
            diff = query - node.point
            dist2 = float(np.dot(diff, diff))
            if dist2 < best["dist2"]:
                best["dist2"] = dist2
                best["node"] = node

            axis = node.axis
            delta = query[axis] - node.point[axis]
            near_branch, far_branch = (node.left, node.right) if delta < 0 else (node.right, node.left)

            _search(near_branch)
            # Critério de poda: só visita o outro lado se o hiperplano estiver dentro do raio atual
            if (delta * delta) < best["dist2"]:
                _search(far_branch)

        _search(self.root)
        return best["node"], float(np.sqrt(best["dist2"])), best["visited"]

    def k_nearest_neighbors(self, query: np.ndarray, k_nn: int = 3) -> List[Tuple[float, Any, np.ndarray]]:
        """Retorna os k_nn vizinhos mais próximos ordenados por distância crescente."""
        query = np.asarray(query, dtype=float)
        heap: List[Tuple[float, int, KDNode]] = []  # max-heap simulada com -dist2
        counter = 0

        def _search(node: Optional[KDNode]):
            nonlocal counter
            if node is None:
                return
            diff = query - node.point
            dist2 = float(np.dot(diff, diff))
            counter += 1
            if len(heap) < k_nn:
                heapq.heappush(heap, (-dist2, counter, node))
            elif dist2 < -heap[0][0]:
                heapq.heapreplace(heap, (-dist2, counter, node))

            axis = node.axis
            delta = query[axis] - node.point[axis]
            near_branch, far_branch = (node.left, node.right) if delta < 0 else (node.right, node.left)

            _search(near_branch)
            if len(heap) < k_nn or (delta * delta) < -heap[0][0]:
                _search(far_branch)

        _search(self.root)
        res = [(float(np.sqrt(-d2)), n.label, n.point) for d2, _, n in heap]
        res.sort(key=lambda x: x[0])
        return res

    def range_search(self, bounds: List[Tuple[float, float]]) -> List[Tuple[np.ndarray, Any]]:
        """Retorna todos os pontos contidos no hiper-retângulo bounds = [(xmin, xmax), ...]."""
        encontrados = []

        def _search(node: Optional[KDNode]):
            if node is None:
                return
            if all(bounds[d][0] <= node.point[d] <= bounds[d][1] for d in range(self.k)):
                encontrados.append((node.point, node.label))
            axis = node.axis
            if bounds[axis][0] <= node.point[axis]:
                _search(node.left)
            if bounds[axis][1] >= node.point[axis]:
                _search(node.right)

        _search(self.root)
        return encontrados

print("✅ Classe KDTree compilada com sucesso.")

---
## 2. Aplicação 1 (2D Espacial): Indexação do Layout da FMS e Aceleração do `AGV-300`

Utilizamos as coordenadas físicas exatas da célula $\mathcal{C} = [0, 5{,}0\text{ m}] \times [0, 4{,}0\text{ m}]$ (Aula 22) para indexar as estações e *waypoints* de navegação do `AGV-300`.

In [ ]:
estacoes_fms = [
    (np.array([0.50, 3.40]), "CLP-00"),
    (np.array([0.70, 2.00]), "SILO-100"),
    (np.array([1.70, 2.00]), "INSP-200"),
    (np.array([2.40, 0.55]), "CX-301 (Vermelha)"),
    (np.array([3.10, 0.55]), "CX-302 (Verde)"),
    (np.array([3.80, 0.55]), "CX-303 (Azul)"),
    (np.array([4.50, 2.00]), "DESC-304"),
    (np.array([4.50, 3.40]), "EXP-400"),
]

kdtree_estacoes = KDTree(estacoes_fms, k=2)

# Consulta: AGV-300 na posição (2.85, 1.15) buscando a estação mais próxima
pos_agv = np.array([2.85, 1.15])
no_prox, dist_prox, visitados = kdtree_estacoes.nearest_neighbor(pos_agv)

# Auditoria contra Força Bruta
melhor_fb = min(estacoes_fms, key=lambda item: np.linalg.norm(item[0] - pos_agv))
assert no_prox.label == melhor_fb[1], "Divergência entre KD-Tree e Força Bruta!"
assert np.isclose(dist_prox, np.linalg.norm(melhor_fb[0] - pos_agv))

# Busca por alcance: estações na zona sul de loteamento (X in [2.0, 4.0], Y in [0.0, 1.5])
zona_loteamento = kdtree_estacoes.range_search([(2.0, 4.0), (0.0, 1.5)])

print(f"📍 Posição atual AGV-300: {pos_agv}")
print(f"🎯 Estação mais próxima: {no_prox.label} | Distância: {dist_prox:.3f} m | Nós visitados: {visitados}/{len(estacoes_fms)}")
print(f"📦 Estações na zona sul de loteamento: {[lbl for _, lbl in zona_loteamento]}")

In [ ]:
# Visualização dos hiperplanos de corte 2D e do círculo de busca/poda
fig, ax = plt.subplots(figsize=(10, 7))

def plot_splits(node: Optional[KDNode], xmin: float, xmax: float, ymin: float, ymax: float):
    if node is None:
        return
    x, y = node.point
    if node.axis == 0:
        ax.plot([x, x], [ymin, ymax], color="#94a3b8", linestyle="--", linewidth=1.2)
        plot_splits(node.left, xmin, x, ymin, ymax)
        plot_splits(node.right, x, xmax, ymin, ymax)
    else:
        ax.plot([xmin, xmax], [y, y], color="#cbd5e1", linestyle=":", linewidth=1.2)
        plot_splits(node.left, xmin, xmax, ymin, y)
        plot_splits(node.right, xmin, xmax, y, ymax)

plot_splits(kdtree_estacoes.root, 0.0, 5.0, 0.0, 4.0)

# Plot das estações
for pt, lbl in estacoes_fms:
    ax.scatter(pt[0], pt[1], s=90, color="#1d4ed8", zorder=4)
    ax.text(pt[0] + 0.07, pt[1] + 0.07, lbl, fontsize=9, fontweight="bold")

# Plot da consulta AGV-300 e raio de busca
ax.scatter(pos_agv[0], pos_agv[1], s=130, color="#dc2626", marker="*", zorder=5, label="Consulta AGV-300")
circulo = patches.Circle((pos_agv[0], pos_agv[1]), dist_prox, fill=False, edgecolor="#dc2626", linestyle="-", linewidth=1.8, label="Raio de Poda (r_best)")
ax.add_patch(circulo)

ax.set_xlim(0, 5.0)
ax.set_ylim(0, 4.0)
ax.set_xlabel("Coordenada X [m]")
ax.set_ylabel("Coordenada Y [m]")
ax.set_title("Particionamento Espacial 2D da KD-Tree no Layout da Célula FMS", fontweight="bold")
ax.legend(loc="upper center")
plt.tight_layout()
plt.show()

---
## 3. Aplicação 2 (3D RGB): Classificação Óptica de Peças no Túnel `INSP-200`

Em operação real, o sensor de visão do túnel `INSP-200` (`AS-201..203`) lê vetores contínuos $(R, G, B) \in [0, 255]^3$ sujeitos a ruído de iluminação. Indexamos um banco calibrado de referências em uma KD-Tree 3D e classificamos novas peças por $k$-NN ($k_{\text{nn}} = 3$).

In [ ]:
from collections import Counter

# Banco calibrado de amostras RGB no túnel INSP-200
amostras_rgb = []
centros_cores = {
    "VERMELHA -> XV-201 / CX-301": np.array([220, 35, 35]),
    "VERDE -> XV-202 / CX-302":    np.array([35, 200, 45]),
    "AZUL -> XV-203 / CX-303":     np.array([30, 55, 225]),
    "REFUGO -> DESC-304":          np.array([120, 120, 120]),
}

for rotulo, centro in centros_cores.items():
    for _ in range(25):
        amostra = np.clip(centro + np.random.normal(0, 12, size=3), 0, 255)
        amostras_rgb.append((amostra, rotulo))

kdtree_rgb = KDTree(amostras_rgb, k=3)

leituras_teste = [
    np.array([212, 42, 30]),
    np.array([40, 192, 50]),
    np.array([25, 62, 218]),
    np.array([128, 115, 122]),
]

resultados_insp = []
for leitura in leituras_teste:
    vizinhos = kdtree_rgb.k_nearest_neighbors(leitura, k_nn=3)
    votos = Counter(lbl for _, lbl, _ in vizinhos)
    classe_predita = votos.most_common(1)[0][0]
    resultados_insp.append({
        "Leitura_RGB": tuple(leitura.astype(int)),
        "Dist_Min": round(vizinhos[0][0], 2),
        "Decisao_Triagem": classe_predita,
    })

pd.DataFrame(resultados_insp)

---
## 4. Benchmark: Busca Linear $O(N)$ vs. KD-Tree $O(\log N)$ na Primitiva `NEAREST` do RRT

In [ ]:
tamanhos = [100, 500, 1000, 2500, 5000, 10000]
resultados_bench = []
consultas = np.column_stack([np.random.uniform(0, 5.0, 50), np.random.uniform(0, 4.0, 50)])

for n in tamanhos:
    pts_np = np.column_stack([np.random.uniform(0, 5.0, n), np.random.uniform(0, 4.0, n)])
    pts_lista = [(pts_np[i], i) for i in range(n)]
    tree = KDTree(pts_lista, k=2)

    # Tempo Força Bruta
    t0 = time.perf_counter()
    for q in consultas:
        dists = np.sum((pts_np - q) ** 2, axis=1)
        idx_fb = int(np.argmin(dists))
    t_fb = (time.perf_counter() - t0) * 1000.0

    # Tempo KD-Tree + média de nós visitados
    t0 = time.perf_counter()
    total_vis = 0
    for q in consultas:
        no_kd, _, vis = tree.nearest_neighbor(q)
        total_vis += vis
    t_kd = (time.perf_counter() - t0) * 1000.0

    # Auditoria na última consulta
    assert no_kd.label == idx_fb, f"Erro de exatidão para N={n}"

    resultados_bench.append({
        "N_Waypoints": n,
        "Tempo_ForcaBruta_ms": round(t_fb, 3),
        "Tempo_KDTree_ms": round(t_kd, 3),
        "Nos_Visitados_Medio": round(total_vis / len(consultas), 1),
        "Taxa_Poda_%": round(100.0 * (1.0 - (total_vis / len(consultas)) / n), 2),
    })

df_bench = pd.DataFrame(resultados_bench)
print(df_bench.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(df_bench["N_Waypoints"], df_bench["N_Waypoints"], "o--", label="Força Bruta (N avaliações)", color="#dc2626")
ax.plot(df_bench["N_Waypoints"], df_bench["Nos_Visitados_Medio"], "s-", label="KD-Tree (Nós visitados médios)", color="#16a34a", linewidth=2)
ax.set_yscale("log")
ax.set_title("Escalabilidade da Primitiva NEAREST: Força Bruta vs. KD-Tree (Escala Log)", fontweight="bold")
ax.set_xlabel("Número de Waypoints RRT (N)")
ax.set_ylabel("Nós Avaliados por Consulta (log)")
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Conclusões de Engenharia

1. **Ganho Expressivo na Navegação do `AGV-300`:** Para $N = 10{.}000$ *waypoints*, a poda *branch-and-bound* da KD-Tree descarta mais de $99\%$ dos nós sem visitá-los, mantendo a exatidão matemática da distância euclidiana.
2. **Versatilidade Multidimensional:** A mesma estrutura `KDTree` atendeu tanto o espaço geométrico 2D da planta quanto o espaço de cores 3D $(R, G, B)$ do túnel óptico `INSP-200`.